In [ ]:
!nvidia-smi
!pip install -q scikit-image

import torch
print("GPU Available:", torch.cuda.is_available())

Thu Oct 15 05:26:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
os.makedirs("src", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("Directories created!")

Directories created!


In [ ]:
%%writefile src/model.py
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.net(x)

class ColorizationNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.inc = DoubleConv(1, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))

        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(128, 64)

        self.outc = nn.Conv2d(64, 2, kernel_size=1)
        self.tanh = nn.Tanh()

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        x = self.up1(x4)
        x = torch.cat([x, x3], dim=1)
        x = self.conv1(x)

        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv2(x)

        x = self.up3(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv3(x)

        return self.tanh(self.outc(x))

Writing src/model.py


In [ ]:
%%writefile src/color_utils.py
import numpy as np
import torch
from PIL import Image
from skimage.color import rgb2lab, lab2rgb

IMAGE_SIZE = 256

def pil_to_lab_tensors(image, size=IMAGE_SIZE):
    image = image.convert("RGB").resize((size, size), Image.Resampling.BILINEAR)
    rgb = np.asarray(image, dtype=np.float32) / 255.0
    lab = rgb2lab(rgb).astype(np.float32)
    L = (lab[..., 0] / 50.0) - 1.0
    ab = np.clip(lab[..., 1:] / 128.0, -1.0, 1.0)
    return torch.from_numpy(L).unsqueeze(0), torch.from_numpy(ab).permute(2, 0, 1)

def lab_to_pil(L, ab):
    if torch.is_tensor(L): L = L.detach().cpu().numpy()
    if torch.is_tensor(ab): ab = ab.detach().cpu().numpy()

    L = (np.squeeze(L) + 1.0) * 50.0
    ab = np.transpose(np.squeeze(ab), (1, 2, 0)) * 128.0

    lab = np.concatenate([L[..., None], ab], axis=2).astype(np.float32)
    lab[..., 0] = np.clip(lab[..., 0], 0, 100)
    lab[..., 1:] = np.clip(lab[..., 1:], -128, 127)

    rgb = lab2rgb(lab)
    rgb = np.clip(rgb, 0, 1)
    return Image.fromarray((rgb * 255).round().astype(np.uint8))

Writing src/color_utils.py


In [ ]:
%%writefile train.py
import argparse, os, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.datasets import OxfordIIITPet
from torchvision.models import vgg16, VGG16_Weights
from src.color_utils import IMAGE_SIZE, pil_to_lab_tensors
from src.model import ColorizationNet

class PetDataset(Dataset):
    def __init__(self, split="trainval", max_samples=None):
        self.ds = OxfordIIITPet(root="data", split=split, target_types="category", download=True)
        self.n = min(max_samples, len(self.ds)) if max_samples else len(self.ds)
    def __len__(self): return self.n
    def __getitem__(self, i):
        image, _ = self.ds[i]
        return pil_to_lab_tensors(image, IMAGE_SIZE)

class VggPerceptualLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.DEFAULT).features[:16].to(device)
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg.eval()

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device))

    def lab_to_rgb(self, L, ab):
        # Convert L from [-1, 1] back to [0, 100]
        L_unnorm = (L + 1.0) * 50.0
        # Convert ab from [-1, 1] back to [-128, 127]
        ab_unnorm = ab * 128.0

        # Convert Lab to RGB mathematically in PyTorch to keep gradients differentiable
        # Reference standard Lab -> XYZ -> sRGB conversion
        # XYZ conversion
        fY = (L_unnorm + 16.0) / 116.0
        fX = ab_unnorm[:, 0:1, :, :] / 500.0 + fY
        fZ = fY - ab_unnorm[:, 1:2, :, :] / 200.0

        def f_inv(t):
            mask = t > 6.0/29.0
            out = torch.zeros_like(t)
            out[mask] = t[mask] ** 3
            out[~mask] = (3.0 * (6.0/29.0)**2) * (t[~mask] - 4.0/29.0)
            return out

        X = f_inv(fX) * 0.950489
        Y = f_inv(fY) * 1.000000
        Z = f_inv(fZ) * 1.088840

        # XYZ to RGB conversion matrix (D65 observer)
        R = X * 3.2406 + Y * -1.5372 + Z * -0.4986
        G = X * -0.9689 + Y * 1.8758 + Z * 0.0415
        B = X * 0.0557 + Y * -0.2040 + Z * 1.0570

        rgb = torch.cat([R, G, B], dim=1)
        rgb = torch.clamp(rgb, 0.0, 1.0)
        return rgb

    def forward(self, pred_ab, target_ab, L_channel):
        pred_rgb = self.lab_to_rgb(L_channel, pred_ab)
        target_rgb = self.lab_to_rgb(L_channel, target_ab)

        pred_feats = self.vgg((pred_rgb - self.mean) / self.std)
        target_feats = self.vgg((target_rgb - self.mean) / self.std)

        return F.l1_loss(pred_feats, target_feats)

def run_epoch(model, loader, device, perceptual_loss_fn, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total = 0.0
    with torch.set_grad_enabled(training):
        for L, ab in loader:
            L, ab = L.to(device), ab.to(device)
            if training: optimizer.zero_grad(set_to_none=True)

            pred = model(L)
            reg_loss = F.smooth_l1_loss(pred, ab)
            perc_loss = perceptual_loss_fn(pred, ab, L)
            loss = reg_loss + 0.2 * perc_loss

            if training:
                loss.backward()
                optimizer.step()
            total += loss.item()
    return total / max(len(loader), 1)

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--epochs", type=int, default=15)
    p.add_argument("--batch-size", type=int, default=16)
    p.add_argument("--learning-rate", type=float, default=0.0005)
    p.add_argument("--max-train", type=int, default=None)
    args, unknown = p.parse_known_args()

    random.seed(42); torch.manual_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs("outputs", exist_ok=True)
    print(f"Device: {device}; image size: {IMAGE_SIZE}x{IMAGE_SIZE}")

    ds = PetDataset(max_samples=args.max_train)
    val_n = max(1, int(len(ds) * 0.1))
    train_n = len(ds) - val_n
    train_ds, val_ds = random_split(ds, [train_n, val_n], generator=torch.Generator().manual_seed(42))

    train_dl = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=2)
    val_dl = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=2)

    model = ColorizationNet().to(device)
    perceptual_loss_fn = VggPerceptualLoss(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.learning_rate)

    best = float("inf")
    for epoch in range(args.epochs):
        tr = run_epoch(model, train_dl, device, perceptual_loss_fn, optimizer)
        va = run_epoch(model, val_dl, device, perceptual_loss_fn)
        print(f"Epoch {epoch+1}/{args.epochs}: train loss={tr:.5f}, validation loss={va:.5f}")
        if va < best:
            best = va
            torch.save(model.state_dict(), "outputs/colorizer.pth")
            print("Saved outputs/colorizer.pth")

    print("Done training!")

if __name__ == "__main__":
    main()

Writing train.py


In [ ]:
!python train.py --epochs 5

Device: cuda; image size: 256x256
100% 19.2M/19.2M [00:01<00:00, 16.2MB/s]
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 528M/528M [00:08<00:00, 68.4MB/s]
Epoch 1/5: train loss=0.09453, validation loss=0.08869
Saved outputs/colorizer.pth
Epoch 2/5: train loss=0.08646, validation loss=0.08793
Saved outputs/colorizer.pth
Epoch 3/5: train loss=0.08578, validation loss=0.08932
Epoch 4/5: train loss=0.08547, validation loss=0.08831
Epoch 5/5: train loss=0.08512, validation loss=0.08714
Saved outputs/colorizer.pth
Done training!


In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from src.model import ColorizationNet
from src.color_utils import IMAGE_SIZE, pil_to_lab_tensors, lab_to_pil
from torchvision.datasets import OxfordIIITPet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ColorizationNet().to(device)
model.load_state_dict(torch.load("outputs/colorizer.pth", map_location=device))
model.eval()

ds = OxfordIIITPet(root="data", split="trainval", download=True)
img, _ = ds[20]

# Save original
orig_resized = img.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
orig_resized.save("outputs/sample_original.png")

# Save grayscale
L, true_ab = pil_to_lab_tensors(img, IMAGE_SIZE)
gray_np = ((np.squeeze(L.numpy()) + 1.0) * 127.5).astype(np.uint8)
Image.fromarray(gray_np).save("outputs/sample_gray.png")

# Save colorized
with torch.no_grad():
    ab_pred = model(L.unsqueeze(0).to(device)).squeeze(0).cpu()
colorized_img = lab_to_pil(L, ab_pred)
colorized_img.save("outputs/sample_colorized.png")

print("Saved sample images to outputs/ successfully!")

Saved sample images to outputs/ successfully!


In [ ]:
%%writefile evaluate.py
import argparse, torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision.datasets import OxfordIIITPet
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
from src.color_utils import IMAGE_SIZE, pil_to_lab_tensors, lab_to_pil
from src.model import ColorizationNet

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ColorizationNet().to(device)
    model.load_state_dict(torch.load("outputs/colorizer.pth", map_location=device))
    model.eval()

    ds = OxfordIIITPet(root="data", split="trainval", download=True)

    mae_list = []
    psnr_list = []
    ssim_list = []

    print("Evaluating model (including SSIM)...")
    with torch.no_grad():
        for i in range(min(100, len(ds))):
            img, _ = ds[i]
            img_resized = img.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
            true_rgb = np.array(img_resized, dtype=np.float32) / 255.0

            L, true_ab = pil_to_lab_tensors(img, IMAGE_SIZE)
            L_dev = L.unsqueeze(0).to(device)

            pred_ab = model(L_dev).squeeze(0).cpu()

            mae = torch.mean(torch.abs(pred_ab - true_ab)).item()
            mae_list.append(mae)

            pred_img = np.array(lab_to_pil(L, pred_ab), dtype=np.float32) / 255.0

            psnr = compute_psnr(true_rgb, pred_img, data_range=1.0)
            psnr_list.append(psnr)

            # Compute SSIM across channels (multichannel=True for RGB images)
            ssim = compute_ssim(true_rgb, pred_img, data_range=1.0, channel_axis=-1)
            ssim_list.append(ssim)

    print(f"--- Evaluation Results ---")
    print(f"Mean a/b MAE: {np.mean(mae_list):.4f}")
    print(f"Mean PSNR: {np.mean(psnr_list):.4f} dB")
    print(f"Mean SSIM: {np.mean(ssim_list):.4f}")

if __name__ == "__main__":
    main()

Writing evaluate.py


In [ ]:
!python evaluate.py

Evaluating model (including SSIM)...
--- Evaluation Results ---
Mean a/b MAE: 0.0770
Mean PSNR: 23.8356 dB
Mean SSIM: 0.9058


In [ ]:
!pip install -q streamlit
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 113.4 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 1s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import numpy as np
import io
from PIL import Image
from src.model import ColorizationNet
from src.color_utils import IMAGE_SIZE, pil_to_lab_tensors, lab_to_pil

@st.cache_resource
def load_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ColorizationNet().to(device)
    model.load_state_dict(torch.load("outputs/colorizer.pth", map_location=device))
    model.eval()
    return model, device

st.title("Automatic Image Colorization (U-Net + VGG)")
st.write("Upload a black-and-white or historical photo to predict plausible colors.")

uploaded_file = st.file_uploader("Upload a JPG or PNG photo", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    input_image = Image.open(uploaded_file).convert("RGB")

    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Grayscale Input (L Channel)")
        L, _ = pil_to_lab_tensors(input_image, IMAGE_SIZE)
        gray_display = Image.fromarray(((np.squeeze(L.numpy()) + 1.0) * 127.5).astype(np.uint8))
        st.image(gray_display, use_container_width=True)

    model, device = load_model()
    with torch.no_grad():
        pred_ab = model(L.unsqueeze(0).to(device)).squeeze(0).cpu()

    # Boost color saturation slightly for presentation demo visibility
    pred_ab = pred_ab * 5
    pred_ab = torch.clamp(pred_ab, -1.0, 1.0)

    colorized_img = lab_to_pil(L, pred_ab)

    with col2:
        st.subheader("Colorized Output")
        st.image(colorized_img, use_container_width=True)

        # Convert colorized PIL image to bytes for correct downloading
        buf = io.BytesIO()
        colorized_img.save(buf, format="PNG")
        byte_im = buf.getvalue()

        st.download_button(
            label="Download Colorized PNG",
            data=byte_im,
            file_name="colorized_output.png",
            mime="image/png"
        )

Overwriting app.py


In [ ]:
# Stop existing background jobs
!pkill -f streamlit

# Run streamlit in the background
import subprocess, time
subprocess.Popen(["streamlit", "run", "app.py", "--server.headless=true", "--server.enableCORS=false", "--server.enableXsrfProtection=false"])
time.sleep(3)

# Use Google Colab's built-in notebook proxy extension
from google.colab.output import eval_js
proxy_url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"🔗 Open your app instantly here: {proxy_url}")

🔗 Open your app instantly here: https://8501-gpu-t4-s-kkb-use1c1-1n1jr53ehmmhz-c.us-east1-1.prod.colab.dev
